---

<h3><center>E7 -  Introduction to Programming for Scientists and Engineers</center></h3>

<h2><center>Discussion week 12<br></center></h2>

<h1><center>Solving nonlinear systems of equations with FPI<br></center></h1>

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

We wish to compute the solutions to the following system on nonlinear equations.
\begin{align*}
10 \xi_1 + \xi_2  &= 1- \sin(\xi_1+\xi_2) \\
-\xi_1 +10 \xi_2 &= 2+ \cos(\xi_1-\xi_2)
\end{align*}

Stated differently, we wish to find the roots of the function $f:\mathbb{R}^2\rightarrow\mathbb{R}^2$ given by
\begin{equation*}
 (\xi_1,\xi_2)  \mapsto (\; 10 \xi_1 + \xi_2 - 1+ \sin(\xi_1+\xi_2)  \;,\; -\xi_1 +10 \xi_2 - 2- \cos(\xi_1-\xi_2)\;)
 \end{equation*}

In [ ]:
def f(xi):
    xi1, xi2 = xi
    return 10*xi1 + xi2 - 1 + np.sin(xi1+xi2) , -xi1 + 10*xi2 -2 - np.cos(xi1-xi2)

## Evaluate $f$ on a grid and see where it evaluates near (0,0)

In [ ]:
# grid of points in the plane
N = 300
xi1 = np.linspace(-0.5,0.5,N)
xi2 = np.linspace(-0.5,0.5,N)
Xi1,Xi2 = np.meshgrid(xi1,xi2)

# Evaluate f on the grid
f1, f2 = f((Xi1,Xi2))

# Tag points that evaluate near (0,0)
isroot = np.logical_and( abs(f1)<0.1 ,  abs(f2)<0.1 )

# Plot
fig, ax = plt.subplots()
ax.contour(xi1,xi2,isroot,cmap='hsv')
ax.spines[['right','top']].set_visible(False)
ax.spines[['bottom','left']].set_position('zero')

ax.set_xlabel(r'$\xi_1$',fontsize=14)
ax.set_ylabel(r'$\xi_2$',fontsize=14,rotation=0)
ax.xaxis.set_label_coords(1.05,0.52)
ax.yaxis.set_label_coords(0.5,1.0)

ax.set_aspect('equal')
ax.grid()

## Approximate location of the root

In [ ]:
eyeball_root = (Xi1[isroot].mean(),Xi2[isroot].mean())

## Create functions for running and plotting FPI

In [ ]:

def fpi(g,xy0,maxsteps,eps):
    XY = np.empty((2,maxsteps))
    XY[:,0] = xy0
    converged = False
    fixedpoint = None
    for k in range(1,maxsteps):
        XY[:,k]=g(XY[:,k-1])
        norm = np.sqrt(np.sum((XY[:,k]-XY[:,k-1])**2))
        if norm<eps:
            converged = True
            fixedpoint = XY[:,k]
    return converged,fixedpoint

def run_fpi_on_grid(g,FPI_ic,maxsteps=30,eps=0.01):
    K = FPI_ic.shape[1]
    converged = np.empty(K)
    xystar = np.empty((2,K))
    
    for k in range(K):
        converged[k],xystar[:,k] = fpi(g,FPI_ic[:,k],maxsteps,eps)

    return converged, xystar

def plotit(converged,fixedpoints,FPI_ic):
    K = FPI_ic.shape[1]
    _, ax = plt.subplots()
    for k in range(K):
        if converged[k]:
            ax.plot(FPI_ic[0,k],FPI_ic[1,k],'go',markersize=2)
            ax.plot([FPI_ic[0,k],fixedpoints[0,k]],[FPI_ic[1,k],fixedpoints[1,k]],color='gray',linewidth=0.5)
            
    ax.plot(eyeball_root[0],eyeball_root[1],'ro')
    ax.xaxis.set_ticks_position('bottom')
    ax.set_xlabel(r'$\xi_1$',fontsize=14)
    ax.set_ylabel(r'$\xi_2$',fontsize=14,rotation=0)
    ax.spines[['right','top']].set_visible(False)
    ax.grid()
    return ax

## Create a grid of initial conditions in $(\xi_1,\xi_2)$

In [ ]:
xi1 = np.linspace(-2,2,30)
xi2 = np.linspace(-2,2,30)
Xi1o,Xi2o = np.meshgrid(xi1,xi2)

FPI_ic = np.vstack((Xi1o.flatten(),Xi2o.flatten()))

# Run and plot FPI using $g_A$

In [ ]:
def gA(xi):
    xi1, xi2 = xi
    return 0.1*(1-xi2-np.sin(xi1+xi2)) , 0.1*(2+xi1+np.cos(xi1-xi2))

converged, fixedpoints = run_fpi_on_grid(gA,FPI_ic,maxsteps=3)

ax = plotit(converged,fixedpoints,FPI_ic)

# Prepare to run FPI using $g_B(\eta_1,\eta_2)$

In [ ]:
def gB(eta):
    eta1, eta2 = eta
    return (-9*eta2+2-2*np.sin(eta1))/11 , (9*eta1-4-2*np.cos(eta2))/11

# transformation xi -> eta
def xi2eta(xi):
    xi1 = xi[0,:]
    xi2 = xi[1,:]
    eta = np.empty(xi.shape)
    eta[0,:] = xi1+xi2
    eta[1,:] = xi1-xi2
    return eta

# transformation eta -> xi
def eta2xi(eta):
    eta1 = eta[0,:]
    eta2 = eta[1,:]
    xi = np.empty(eta.shape)
    xi[0,:] = (eta1+eta2)/2
    xi[1,:] = (eta1-eta2)/2
    return xi


# Run and plot FPI using $g_B$

In [ ]:
eta_ic = xi2eta(FPI_ic)
converged, fp_eta = run_fpi_on_grid(gB,eta_ic,maxsteps=3)

fp_xi = eta2xi(fp_eta)

ax = plotit(converged,fp_xi,FPI_ic)